# Garcia - step 1: AnnData preparation

This notebook computes everything about `garcia` that is expensive and
stable, and writes it once to an intermediate `.h5ad`:

- one shared k-nearest-neighbour graph on the scVI latent representation;
- 1D, 2D and 3D UMAP embeddings, all three from **that one graph**, so the three
  views of the dataset encode identical neighbour relationships;
- the latent representation the viewer uses for centroid statistics;
- the curated cell metadata.

Step 2 (`garcia_2_export_cellucid.ipynb`) reads only the intermediate, resolves
the gene names, and calls `cellucid.prepare`. The split exists so that a change to
the export format, to the gene naming, or to the `prepare` API costs one cheap
re-run of step 2 instead of a full re-embedding.

**This notebook establishes no gene names.** The source `var` index is the
positional matrix column number and nothing else; the identifiers live in
`.anndatas/varnames/dataset_complete_Garcia_varnames.pickle`, which step 2
reads and resolves. Nothing here reads `exports/`, so this notebook is re-runnable
against the source alone and cannot be invalidated by republishing the dataset.

The historical `prepare_garcia.ipynb` that shipped beside the source data is the
`he` recipe with two strings changed, so this notebook is the `he` notebook with
the same two strings changed.

Run this notebook once. Re-run it only when the source data, the graph parameters,
or the cell metadata change.


## Environment

The interpreter needs `scanpy` and `umap-learn`. Nothing here writes inside the
source data directory.


In [1]:
import gc
import os
import resource
import sys
import time
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

sc.settings.verbosity = 1

# scanpy reaches for tqdm's notebook frontend, which warns when ipywidgets is
# absent. It concerns progress-bar rendering only, and the warning text carries
# the interpreter's own install path into the committed output.
warnings.filterwarnings("ignore", message="IProgress not found")

TIMINGS: dict[str, float] = {}

# getrusage reports the high-water mark in bytes on macOS and in kibibytes
# everywhere else.
RSS_UNIT = 1 if sys.platform == "darwin" else 1024


def peak_rss_gb() -> float:
    """Peak resident set size of this kernel so far, in GB."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * RSS_UNIT / 1e9


@contextmanager
def stage(name: str):
    """Record the wall-clock cost of one preparation stage."""
    start = time.perf_counter()
    yield
    TIMINGS[name] = time.perf_counter() - start
    print(f"[{name}] {TIMINGS[name]:.1f} s  peak RSS {peak_rss_gb():.2f} GB")


ANNDATA_VERSION = version("anndata")
SCANPY_VERSION = version("scanpy")
NUMPY_VERSION = version("numpy")

print("anndata", ANNDATA_VERSION)
print("scanpy ", SCANPY_VERSION)
print("numpy  ", NUMPY_VERSION)


anndata 0.12.19
scanpy  1.11.5
numpy   1.26.4


## Paths

Both notebooks locate the repository by walking up for `exports/datasets.json`,
so they run from any working directory.

- **Source** - `.anndatas/garcia.h5ad` beside the repository, read-only.
  Override with `CELLUCID_ANNDATA_DIR`.
- **Intermediate** - written *outside every repository*. It carries the full
  count matrix and is far too large to track in git. Override with
  `CELLUCID_INTERMEDIATE_DIR`.


In [2]:
def find_repository_root(start: Path) -> Path:
    """Return the cellucid-datasets checkout that contains `start`."""
    for candidate in [start, *start.parents]:
        if (candidate / "exports" / "datasets.json").is_file():
            return candidate
    raise RuntimeError(f"No cellucid-datasets checkout above {start}")


DATASET_ID = "garcia"

REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
WORKSPACE_ROOT = REPOSITORY_ROOT.parent

SOURCE_DIR = Path(os.environ.get("CELLUCID_ANNDATA_DIR", WORKSPACE_ROOT / ".anndatas"))
INTERMEDIATE_DIR = Path(
    os.environ.get("CELLUCID_INTERMEDIATE_DIR", WORKSPACE_ROOT / ".anndatas-intermediate")
)

SOURCE_FILE = SOURCE_DIR / f"{DATASET_ID}.h5ad"
INTERMEDIATE_FILE = INTERMEDIATE_DIR / f"{DATASET_ID}_prepared.h5ad"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)


def show(path: Path) -> str:
    """Render a path relative to the workspace so committed output is portable."""
    try:
        return str(path.relative_to(WORKSPACE_ROOT))
    except ValueError:
        return str(path)


print("repository  :", show(REPOSITORY_ROOT))
print("source      :", show(SOURCE_FILE), f"({SOURCE_FILE.stat().st_size / 1e6:.1f} MB)")
print("intermediate:", show(INTERMEDIATE_FILE))


repository  : cellucid-datasets
source      : .anndatas/garcia.h5ad (697.3 MB)
intermediate: .anndatas-intermediate/garcia_prepared.h5ad


## Graph and embedding parameters

One seed and one neighbour count govern the graph and all three UMAPs. These are
not free choices: they are the values the original build used, read out of the
historical `prepare_garcia.ipynb` that shipped beside the source data.
Changing any of them changes the published layout.


In [3]:
LATENT_KEY = "scvi"        # scVI latent representation shipped with the source
N_NEIGHBORS = 15           # neighbours in the one shared kNN graph
MIN_DIST = 0.5             # UMAP min_dist, identical for 1D/2D/3D
RANDOM_SEED = 0            # governs the graph and every embedding

UMAP_DIMENSION_KEYS = {1: "X_umap_1d", 2: "X_umap_2d", 3: "X_umap_3d"}


## Load the source

`garcia.h5ad` is the `garcia_developmental_complete.h5ad` produced by the scVI
integration pipeline and copied by `cellucid-private/scripts/get_data.sh`. The
historical notebook wrote its embeddings into a second file,
`garcia_developmental_complete_with_3d_umap.h5ad`, which was never copied here;
this notebook rebuilds those embeddings from the complete file instead, which is
why nothing downstream depends on the missing one.


In [4]:
with stage("load_source"):
    source = ad.read_h5ad(SOURCE_FILE)

print(source)
print()
print("cells:", f"{source.n_obs:,}", " features:", f"{source.n_vars:,}")

counts = source.X
assert sp.issparse(counts), "expected a sparse count matrix"
assert np.all(counts.data == np.round(counts.data)), "X is not integer-valued"
print(
    "X: sparse counts,",
    f"nnz={counts.nnz:,}",
    f"({counts.nnz / (source.n_obs * source.n_vars):.2%} dense),",
    f"max={counts.data.max():.0f}",
)


[load_source] 0.9 s  peak RSS 0.85 GB
AnnData object with n_obs × n_vars = 219731 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', 'LVL3', 'LVL2', 'LVL1', 'LVL0', '_scvi_batch', '_scvi_labels'
    uns: 'metrics', 'neighbors', 'rank_genes_groups', 'umap'
    obsm: 'Unintegrated', 'X_pca', 'X_umap', 'harmony', 'scvi'
    obsp: 'connectivities', 'distances'

cells: 219,731  features: 8,192


X: sparse counts, nnz=35,337,959 (1.96% dense), max=2578


## Expression values: raw counts, deliberately

`prepare` receives the untransformed integer count matrix. The historical notebook
calls `sc.pp.normalize_total` and `sc.pp.log1p`, but it applies them to a *second*
AnnData object that it then never uses, and the export it produced carries integer
count maxima throughout.

The rebuild keeps the raw integer counts because that is what every other
generation in this catalog contains. Switching the catalog to depth-normalized
expression is a defensible change, but it is a decision about all five datasets at
once.


## Cell metadata

The edits the exported dataset depends on, and nothing else:

- `_scvi_batch` and `_scvi_labels` are scVI bookkeeping. They are integer codes
  with no meaning to a reader and are dropped.
- `age` is stored as a *categorical* of numeric strings. Post-conception weeks are
  an ordered quantity, so it is converted to a float and exported as a continuous
  field; leaving it categorical would forbid ordering, ranges and a continuous
  colour map, and would sort week 10 before week 6.
- A category label that is padded with whitespace has no glyph the legend can
  draw, and `cellucid.prepare` rejects one rather than repairing it: trimming can
  merge two categories and move cells between them without saying so. The decision
  therefore belongs to the dataset build. The scan below runs over every
  categorical field, proves that trimming merges nothing before applying it, and
  renames the *categories* rather than rewriting the values, so every cell keeps
  the code it already had.

Everything else is passed through unchanged.


In [5]:
obs = source.obs.copy()

dropped = [column for column in ("_scvi_batch", "_scvi_labels") if column in obs]
obs = obs.drop(columns=dropped)
print("dropped:", dropped)

age_labels = obs["age"].cat.categories.tolist()
obs["age"] = obs["age"].astype(str).astype(np.float64)
assert obs["age"].notna().all(), "age must convert cleanly to a number"
print(f"\nage: categorical {age_labels} -> continuous "
      f"[{obs['age'].min()}, {obs['age'].max()}]")

categorical = [c for c in obs.columns if isinstance(obs[c].dtype, pd.CategoricalDtype)]
padded = {
    column: [label for label in obs[column].cat.categories
             if isinstance(label, str) and label != label.strip()]
    for column in categorical
}
padded = {column: labels for column, labels in padded.items() if labels}
print(f"\ncategorical fields carrying padded labels: "
      f"{ {column: len(labels) for column, labels in padded.items()} or 'none'}")

for column in padded:
    source_labels = obs[column].cat.categories.tolist()
    trimmed = [label.strip() for label in source_labels]
    # Trimming is only honest if it merges nothing. Two labels collapsing to one
    # would move cells between categories, which is a change to the annotation
    # rather than to its presentation, and this build must not make it silently.
    assert len(set(trimmed)) == len(set(source_labels)), (
        f"stripping {column!r} merges categories and would move cells between them: "
        f"{sorted(set(source_labels))}"
    )
    codes_before = obs[column].cat.codes.to_numpy().copy()
    obs[column] = obs[column].cat.rename_categories(trimmed)
    assert np.array_equal(obs[column].cat.codes.to_numpy(), codes_before), (
        f"trimming {column!r} moved cells between categories"
    )
    print(f"  {column}: {len(padded[column])} of {len(source_labels)} labels trimmed, "
          f"every cell keeps its code")
    print(f"    before {source_labels}")
    print(f"    after  {trimmed}")

LABEL_NORMALIZATION = ", ".join(
    f"{column}: trailing whitespace stripped from {len(labels)} of "
    f"{obs[column].cat.categories.size} labels"
    for column, labels in padded.items()
) or "none"

continuous = [c for c in obs.columns if not isinstance(obs[c].dtype, pd.CategoricalDtype)]
categorical = [c for c in obs.columns if isinstance(obs[c].dtype, pd.CategoricalDtype)]
print(f"\n{len(obs.columns)} obs fields: "
      f"{len(continuous)} continuous {continuous}, {len(categorical)} categorical")
print("largest category count:",
      max(obs[c].cat.categories.size for c in categorical))


dropped: ['_scvi_batch', '_scvi_labels']

age: categorical ['10.0', '11.0', '12.0', '13.0', '14.0', '15.0', '16.0', '17.0', '18.0', '19.0', '21.0', '6.0', '6.5', '7.0', '7.5', '8.0', '8.4', '8.6', '8.8', '9.0'] -> continuous [6.0, 21.0]

categorical fields carrying padded labels: none

16 obs fields: 2 continuous ['age', 'n_genes'], 14 categorical
largest category count: 52


## Feature identity: positional here, named in step 2

**`.anndatas/garcia.h5ad` carries no gene names.** Its `var` index is the
positional matrix column number, `'0'` ... `'8191'`, and `var` has no other column.
The integration pipeline held the identifiers in a separate `var` frame, which is
`.anndatas/varnames/dataset_complete_Garcia_varnames.pickle`.

That frame is read in **step 2**, together with the one permitted symbol mapping,
because naming a gene is a decision about what the viewer prints and not about how
the cells are laid out. Keeping it there means a change to the naming costs one
cheap re-run of step 2 rather than a re-embedding, and it keeps this notebook
dependent on the source file alone.

What this notebook does record is the positional index itself, in
`var['source_var_index']`, so that step 2's alignment of the identifier frame to
the matrix columns is checkable rather than assumed.


In [6]:
var = source.var.copy()
assert list(var.columns) == [], f"source var unexpectedly carries columns: {list(var.columns)}"
assert var.index.tolist() == [str(i) for i in range(source.n_vars)], (
    "source var index is not the positional matrix column number - step 2 aligns "
    "the identifier frame to it positionally and can no longer do so"
)

var["source_var_index"] = np.arange(source.n_vars, dtype=np.int64)
var.index = var.index.astype(str)
var.index.name = "source_var_position"

FEATURE_IDENTITY = "positional_source_index; gene names resolved in step 2"
print("feature identity:", FEATURE_IDENTITY)
print("features:", f"{var.shape[0]:,}", " first:", var.index[:3].tolist(),
      " last:", var.index[-3:].tolist())


feature identity: positional_source_index; gene names resolved in step 2
features: 8,192  first: ['0', '1', '2']  last: ['8189', '8190', '8191']


## One shared kNN graph

The source ships `obsp['connectivities']`, and it is **not** the graph this
build uses. `uns['neighbors']['params']` records `method='rapids'`; RAPIDS is a
GPU library that is not part of this environment, so a recipe that reuses that
graph cannot be re-run. The graph is recomputed with scanpy on the scVI latent
space at the historical parameters, which is also what makes the export
self-consistent: the connectivity edges the viewer draws are edges of the same
graph the embeddings were laid out from.

Above 4,096 cells scanpy's neighbour search is approximate, so the cell below
prints how far the recompute lands from the graph shipped beside the source.


In [7]:
print("source graph:", dict(source.uns["neighbors"]["params"]))

adata = ad.AnnData(
    X=counts,
    obs=obs,
    var=var,
    obsm={LATENT_KEY: np.asarray(source.obsm[LATENT_KEY], dtype=np.float32)},
)
shipped_graph = source.obsp["connectivities"]

with stage("neighbors"):
    sc.pp.neighbors(
        adata,
        n_neighbors=N_NEIGHBORS,
        use_rep=LATENT_KEY,
        random_state=RANDOM_SEED,
    )

connectivities = adata.obsp["connectivities"].tocsr()
connectivities.eliminate_zeros()
assert (abs(connectivities - connectivities.T) > 0).nnz == 0, "graph must be symmetric"
assert connectivities.diagonal().sum() == 0, "graph must have a zero diagonal"
assert connectivities.data.min() > 0, "graph must not store zero weights"
adata.obsp["connectivities"] = connectivities

print("recomputed graph:", dict(adata.uns["neighbors"]["params"]))
print(f"edges (stored, both directions): {connectivities.nnz:,}"
      f"  undirected: {connectivities.nnz // 2:,}")

shipped_upper = sp.triu(shipped_graph, k=1).tocoo()
shipped_keys = shipped_upper.row.astype(np.int64) * adata.n_obs + shipped_upper.col.astype(np.int64)
rebuilt = sp.triu(connectivities, k=1).tocoo()
rebuilt_keys = rebuilt.row.astype(np.int64) * adata.n_obs + rebuilt.col.astype(np.int64)
shared = np.intersect1d(np.sort(rebuilt_keys), np.sort(shipped_keys)).size
print(f"shipped RAPIDS graph: {shipped_keys.size:,} undirected edges, "
      f"{shared:,} of them also in this recompute")

del source, counts, shipped_graph, shipped_upper, shipped_keys, rebuilt, rebuilt_keys
gc.collect()
print(f"source released; peak RSS so far {peak_rss_gb():.2f} GB")


source graph: {'method': 'rapids', 'metric': 'euclidean', 'n_neighbors': 15, 'random_state': 0}


[neighbors] 39.5 s  peak RSS 1.49 GB
recomputed graph: {'n_neighbors': 15, 'method': 'umap', 'random_state': 0, 'metric': 'euclidean', 'use_rep': 'scvi'}
edges (stored, both directions): 5,363,016  undirected: 2,681,508


shipped RAPIDS graph: 2,687,613 undirected edges, 2,463,027 of them also in this recompute
source released; peak RSS so far 1.49 GB


## 1D, 2D and 3D UMAP from that one graph

Each embedding is laid out on a throwaway `AnnData` that carries the same
`obsp`/`uns['neighbors']` pair, so `sc.tl.umap` never rebuilds the graph and the
three layouts stay mutually consistent. Only the number of components differs.


In [8]:
def umap_from_shared_graph(host: ad.AnnData, n_components: int) -> np.ndarray:
    """Lay out the already-computed neighbour graph in `n_components` dimensions."""
    scratch = ad.AnnData(
        obs=host.obs[[]],
        obsp={
            "connectivities": host.obsp["connectivities"],
            "distances": host.obsp["distances"],
        },
        obsm={LATENT_KEY: host.obsm[LATENT_KEY]},
        uns={"neighbors": dict(host.uns["neighbors"])},
    )
    sc.tl.umap(
        scratch,
        n_components=n_components,
        min_dist=MIN_DIST,
        random_state=RANDOM_SEED,
    )
    embedding = np.asarray(scratch.obsm["X_umap"], dtype=np.float32)
    del scratch
    gc.collect()
    return embedding


for n_components, key in UMAP_DIMENSION_KEYS.items():
    with stage(f"umap_{n_components}d"):
        adata.obsm[key] = umap_from_shared_graph(adata, n_components)

for n_components, key in UMAP_DIMENSION_KEYS.items():
    coords = adata.obsm[key]
    assert coords.shape == (adata.n_obs, n_components)
    assert np.isfinite(coords).all()
    spans = coords.max(axis=0) - coords.min(axis=0)
    assert spans.min() > 0, f"{key} collapsed to a point"
    print(f"{key}: shape {coords.shape}, span {np.round(spans, 2).tolist()}")


[umap_1d] 142.3 s  peak RSS 1.75 GB


[umap_2d] 158.0 s  peak RSS 1.87 GB


[umap_3d] 149.2 s  peak RSS 1.87 GB
X_umap_1d: shape (219731, 1), span [46.099998474121094]
X_umap_2d: shape (219731, 2), span [18.34000015258789, 19.239999771118164]
X_umap_3d: shape (219731, 3), span [11.449999809265137, 13.210000038146973, 15.069999694824219]


## Write the intermediate

The intermediate holds exactly what step 2 consumes and nothing else: counts,
curated `obs`, the positional feature index, the latent representation, the three
embeddings, the shared graph, and a provenance record.

`created_at` is fixed here and reused by step 2, so re-running the export after an
API change reproduces the same `dataset_identity.json` timestamp instead of
inventing a new one.


In [9]:
CREATED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

adata.uns = {
    "cellucid_build": {
        "dataset_id": DATASET_ID,
        "created_at": CREATED_AT,
        "source_file": SOURCE_FILE.name,
        "source_bytes": int(SOURCE_FILE.stat().st_size),
        "latent_key": LATENT_KEY,
        "n_neighbors": int(N_NEIGHBORS),
        "min_dist": float(MIN_DIST),
        "random_state": int(RANDOM_SEED),
        "feature_identity": FEATURE_IDENTITY,
        "label_normalization": LABEL_NORMALIZATION,
        "expression": "raw integer counts, untransformed",
        "anndata_version": ANNDATA_VERSION,
        "scanpy_version": SCANPY_VERSION,
        "numpy_version": NUMPY_VERSION,
    }
}

with stage("write_intermediate"):
    adata.write_h5ad(INTERMEDIATE_FILE, compression="gzip")

print(adata)
print()
print("wrote", show(INTERMEDIATE_FILE), f"({INTERMEDIATE_FILE.stat().st_size / 1e6:.1f} MB)")
print("created_at", CREATED_AT)


[write_intermediate] 7.6 s  peak RSS 1.87 GB
AnnData object with n_obs × n_vars = 219731 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    var: 'source_var_index'
    uns: 'cellucid_build'
    obsm: 'scvi', 'X_umap_1d', 'X_umap_2d', 'X_umap_3d'
    obsp: 'distances', 'connectivities'

wrote .anndatas-intermediate/garcia_prepared.h5ad (173.4 MB)
created_at 2026-07-31T11:10:42Z


## Measured cost

Wall-clock per stage and the peak resident set size on the machine that produced
the committed outputs.


In [10]:
total = sum(TIMINGS.values())
for name, seconds in TIMINGS.items():
    print(f"{name:<26} {seconds:8.1f} s")
print(f"{'TOTAL':<26} {total:8.1f} s")
print(f"{'peak RSS':<26} {peak_rss_gb():8.2f} GB")


load_source                     0.9 s
neighbors                      39.5 s
umap_1d                       142.3 s
umap_2d                       158.0 s
umap_3d                       149.2 s
write_intermediate              7.6 s
TOTAL                         497.5 s
peak RSS                       1.87 GB


Next: run `garcia_2_export_cellucid.ipynb`. It reads only
`<intermediate>/garcia_prepared.h5ad`, resolves the gene names, and never
recomputes an embedding.
